# Brain Tumor Classification — Improved VGG16 Transfer Learning

### Key improvements over original:
| Issue | Fix |
|---|---|
| All VGG16 layers frozen | Unfreeze `block5` for fine-tuning |
| No hidden layers | Added Dense(512) → Dense(256) head |
| No regularisation | Added BatchNormalization + Dropout |
| Bug: test used augmentation | `test_datagen` (rescale-only) for test data |
| shuffle=False in training | shuffle=True for training |
| No callbacks | EarlyStopping, ReduceLROnPlateau, ModelCheckpoint |
| Only 10 epochs | Up to 30 epochs with early stopping |
| Default Adam lr (1e-3) | Lower lr (1e-4) suitable for fine-tuning |
| Flatten → huge param count | GlobalAveragePooling2D (fewer params, less overfit) |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from keras.models import Model
from keras.applications.vgg16 import VGG16
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
IMG_SIZE   = [224, 224]
BATCH_SIZE = 32          # increased from 16 → better gradient estimates
EPOCHS     = 30          # early stopping will kick in well before 30
LR         = 1e-4        # lower LR for fine-tuning pretrained weights

In [ ]:
# ── Dataset paths (update as needed) ──────────────────────────────────────────
train_path = '/home/anurag/my-project/1.BT-Restnet/BT1/Training'
test_path  = '/home/anurag/my-project/1.BT-Restnet/BT1/Testing'

In [ ]:
# ── Load pretrained VGG16 (no top) ────────────────────────────────────────────
vgg = VGG16(input_shape=IMG_SIZE + [3], weights='imagenet', include_top=False)

In [ ]:
# ── Fine-tuning strategy ───────────────────────────────────────────────────────
# Freeze all layers first, then unfreeze block5 (last conv block).
# This lets the deepest features adapt to brain MRI scans
# while keeping lower-level edge/texture features stable.

for layer in vgg.layers:
    layer.trainable = False

# Unfreeze the last convolutional block for domain adaptation
for layer in vgg.layers:
    if layer.name.startswith('block5'):
        layer.trainable = True

# Show which layers are trainable
for layer in vgg.layers:
    print(f"{layer.name:30s}  trainable={layer.trainable}")

In [ ]:
# ── Build classification head ─────────────────────────────────────────────────
# GlobalAveragePooling2D replaces Flatten:
#   • Flatten(7×7×512) → 25 088 features → 100 K params in next dense layer
#   • GAP(7×7×512)     →    512 features →   2 K params → much less overfitting

x = GlobalAveragePooling2D()(vgg.output)

x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)   # stabilises training
x = Dropout(0.5)(x)           # strong regularisation on first dense layer

x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)           # lighter regularisation on second layer

prediction = Dense(4, activation='softmax')(x)

In [ ]:
# ── Assemble model ────────────────────────────────────────────────────────────
model = Model(inputs=vgg.input, outputs=prediction)
model.summary()

In [ ]:
# ── Compile ───────────────────────────────────────────────────────────────────
# Adam with a lower learning rate (1e-4) is important when fine-tuning:
# the default 1e-3 can destroy pretrained weights.
model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ── Data generators ───────────────────────────────────────────────────────────
# FIX: original code used train_datagen (with augmentation) for test data → wrong!
# Test/validation data must only be rescaled, never augmented.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=15,           # added: mild rotation
    width_shift_range=0.1,       # added: small translation
    height_shift_range=0.1,      # added: small translation
    brightness_range=[0.8, 1.2]  # added: brightness variation
)

# Only rescale for test/validation — NO augmentation
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# FIX: shuffle=True for training (original had shuffle=False)
train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    shuffle=True,                # FIX: was False
    class_mode='categorical'
)

# FIX: use test_datagen (no augmentation) for test set
test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_mode='categorical'
)

print("Class indices:", train_data.class_indices)

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    # Stop training when val_accuracy stops improving for 5 epochs
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Halve the learning rate when val_loss stalls for 3 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    # Save the best model weights automatically
    ModelCheckpoint(
        'best_brain_tumor_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
r = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# ── Plot: Accuracy ────────────────────────────────────────────────────────────
epochs_ran = range(1, len(r.history['accuracy']) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, r.history['accuracy'],     label='Train Accuracy',      marker='o')
plt.plot(epochs_ran, r.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot: Loss ────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, r.history['loss'],     label='Train Loss',      marker='o')
plt.plot(epochs_ran, r.history['val_loss'], label='Validation Loss', marker='o')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot: Learning rate over time (shows ReduceLROnPlateau effect) ─────────────
if 'lr' in r.history:
    plt.figure(figsize=(8, 3))
    plt.plot(epochs_ran, r.history['lr'], marker='o', color='green')
    plt.title('Learning Rate Schedule')
    plt.xlabel('Epochs')
    plt.ylabel('Learning Rate')
    plt.yscale('log')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Final Evaluation ──────────────────────────────────────────────────────────
loss, accuracy = model.evaluate(test_data)
print(f"\n✅ Test Loss    : {loss:.4f}")
print(f"✅ Test Accuracy: {accuracy*100:.2f}%")